In [1]:
import joblib

# Load final trained artifacts
model = joblib.load("../models/final_isolation_forest.pkl")
scaler = joblib.load("../models/final_scaler.pkl")
threshold = joblib.load("../models/fraud_threshold.pkl")

print("Model loaded successfully")
print("Scaler loaded successfully")
print("Fraud threshold:", threshold)

Model loaded successfully
Scaler loaded successfully
Fraud threshold: 0.03604179957275999


In [2]:
import pandas as pd

# Load the official 2013 dataset
df = pd.read_csv("../dataset/2013 dataset/creditcard.csv")

# Remove duplicates
df = df.drop_duplicates().copy()

# Separate features from target
X = df.drop("Class", axis=1)

print("Dataset loaded successfully")
print("Shape:", X.shape)
print("Features:", list(X.columns))

Dataset loaded successfully
Shape: (283726, 30)
Features: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [3]:
def predict_transaction(transaction):
    """
    Predict whether a credit card transaction is potentially fraudulent.

    Parameters:
        transaction: pandas DataFrame containing one or more transactions.

    Returns:
        DataFrame containing anomaly score and prediction.
    """

    # Make a copy so the original data is not modified
    transaction = transaction.copy()

    # Scale Amount using the saved training scaler
    transaction["Amount"] = scaler.transform(
        transaction[["Amount"]]
    )

    # Generate anomaly score
    decision_score = model.decision_function(transaction)
    anomaly_score = -decision_score

    # Apply the saved fraud threshold
    prediction = (anomaly_score >= threshold).astype(int)

    # Store results
    result = transaction.copy()
    result["anomaly_score"] = anomaly_score
    result["predicted_fraud"] = prediction

    # Convert prediction to readable label
    result["prediction"] = result["predicted_fraud"].map({
        0: "LEGITIMATE",
        1: "FRAUD"
    })

    return result

In [4]:
# Take one transaction from the dataset
test_transaction = X.iloc[[0]].copy()

# Run prediction
result = predict_transaction(test_transaction)

# Display the result
print("Anomaly Score:", result["anomaly_score"].iloc[0])
print("Prediction:", result["prediction"].iloc[0])

Anomaly Score: -0.21784985737187562
Prediction: LEGITIMATE


In [5]:
# Get one known fraudulent transaction from the dataset
fraud_transaction = df[df["Class"] == 1].drop("Class", axis=1).iloc[[0]].copy()

# Run prediction
fraud_result = predict_transaction(fraud_transaction)

print("Anomaly Score:", fraud_result["anomaly_score"].iloc[0])
print("Prediction:", fraud_result["prediction"].iloc[0])
print("Actual Class: FRAUD")

Anomaly Score: -0.08818137287646088
Prediction: LEGITIMATE
Actual Class: FRAUD


In [6]:
# Score all known fraudulent transactions
fraud_transactions = df[df["Class"] == 1].drop("Class", axis=1).copy()

fraud_scores = -model.decision_function(
    fraud_transactions
)

# Find fraud transactions detected by the model
detected_fraud = fraud_transactions[
    fraud_scores >= threshold
].copy()

detected_fraud["anomaly_score"] = fraud_scores[
    fraud_scores >= threshold
]

print("Total known fraud transactions:", len(fraud_transactions))
print("Fraud transactions detected:", len(detected_fraud))
print("Detection rate:", len(detected_fraud) / len(fraud_transactions))

print("\nExample detected fraud:")
print(detected_fraud.head(1))

Total known fraud transactions: 473
Fraud transactions detected: 148
Detection rate: 0.3128964059196617

Example detected fraud:
        Time        V1        V2        V3        V4        V5        V6  \
6971  9064.0 -3.499108  0.258555 -4.489558  4.853894 -6.974522  3.628382   

            V7        V8       V9  ...       V21       V22       V23  \
6971  5.431271 -1.946734 -0.77568  ... -1.052368  0.204817 -2.119007   

           V24       V25       V26       V27       V28   Amount  anomaly_score  
6971  0.170279 -0.393844  0.296367  1.985913 -0.900452  1809.68       0.071676  

[1 rows x 31 columns]


In [7]:
# Take the detected fraud transaction
sample_fraud = df.loc[[6971]].drop("Class", axis=1)

# Run it through the actual prediction function
result = predict_transaction(sample_fraud)

print("Anomaly Score:", result["anomaly_score"].iloc[0])
print("Prediction:", result["prediction"].iloc[0])
print("Threshold:", threshold)

Anomaly Score: 0.06766200450003801
Prediction: FRAUD
Threshold: 0.03604179957275999


In [8]:
def predict_new_transaction(transaction_data):
    """
    Predict a new credit card transaction.

    transaction_data must contain:
    Time, V1-V28, Amount
    """

    # Convert input into DataFrame
    transaction = pd.DataFrame([transaction_data])

    # Check required columns
    required_columns = X.columns.tolist()

    missing_columns = [
        col for col in required_columns
        if col not in transaction.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    # Keep the exact feature order used during training
    transaction = transaction[required_columns]

    # Scale Amount
    transaction["Amount"] = scaler.transform(
        transaction[["Amount"]]
    )

    # Calculate anomaly score
    decision_score = model.decision_function(transaction)
    anomaly_score = -decision_score[0]

    # Make prediction
    prediction = (
        "FRAUD"
        if anomaly_score >= threshold
        else "LEGITIMATE"
    )

    return {
        "anomaly_score": anomaly_score,
        "threshold": threshold,
        "prediction": prediction
    }

In [9]:
# Get one known fraud transaction
sample = df.loc[6971].drop("Class").to_dict()

# Predict using the new prediction interface
result = predict_new_transaction(sample)

print("=" * 35)
print("   CREDIT CARD FRAUD CHECK")
print("=" * 35)

print(f"Anomaly Score : {result['anomaly_score']:.6f}")
print(f"Threshold     : {result['threshold']:.6f}")
print(f"Prediction    : {result['prediction']}")

print("=" * 35)

   CREDIT CARD FRAUD CHECK
Anomaly Score : 0.067662
Threshold     : 0.036042
Prediction    : FRAUD


In [11]:
import mysql.connector

try:
    connection = mysql.connector.connect(
        host="127.0.0.1",
        port=3307,
        user="root",
        password="Fraud@2026",
        database="credit_card_fraud"
    )

    print("MySQL connection successful!")

except mysql.connector.Error as error:
    print("MySQL connection failed:")
    print(error)

MySQL connection successful!


In [14]:
def save_prediction_to_mysql(transaction_data, result):
    cursor = connection.cursor()

    columns = [
        "Time",
        "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9", "V10",
        "V11", "V12", "V13", "V14", "V15", "V16", "V17", "V18", "V19", "V20",
        "V21", "V22", "V23", "V24", "V25", "V26", "V27", "V28",
        "Amount",
        "actual_class",
        "anomaly_score",
        "predicted_fraud"
    ]

    values = [
        transaction_data["Time"],
        transaction_data["V1"],
        transaction_data["V2"],
        transaction_data["V3"],
        transaction_data["V4"],
        transaction_data["V5"],
        transaction_data["V6"],
        transaction_data["V7"],
        transaction_data["V8"],
        transaction_data["V9"],
        transaction_data["V10"],
        transaction_data["V11"],
        transaction_data["V12"],
        transaction_data["V13"],
        transaction_data["V14"],
        transaction_data["V15"],
        transaction_data["V16"],
        transaction_data["V17"],
        transaction_data["V18"],
        transaction_data["V19"],
        transaction_data["V20"],
        transaction_data["V21"],
        transaction_data["V22"],
        transaction_data["V23"],
        transaction_data["V24"],
        transaction_data["V25"],
        transaction_data["V26"],
        transaction_data["V27"],
        transaction_data["V28"],
        transaction_data["Amount"],
        None,
        result["anomaly_score"],
        1 if result["prediction"] == "FRAUD" else 0
    ]

    # Safety check
    assert len(columns) == len(values), (
        f"Column/value mismatch: {len(columns)} columns, "
        f"{len(values)} values"
    )

    placeholders = ", ".join(["%s"] * len(values))

    query = f"""
        INSERT INTO transactions
        ({", ".join(columns)})
        VALUES ({placeholders})
    """

    cursor.execute(query, tuple(values))
    connection.commit()

    print("Prediction saved to MySQL!")
    print("Transaction ID:", cursor.lastrowid)

    cursor.close()

In [15]:
# Get the known fraud transaction
sample_transaction = df.loc[6971].drop("Class").to_dict()

# Run prediction
result = predict_new_transaction(sample_transaction)

# Save prediction to MySQL
save_prediction_to_mysql(sample_transaction, result)

print("\nPrediction result:")
print("Anomaly Score:", result["anomaly_score"])
print("Prediction:", result["prediction"])

Prediction saved to MySQL!
Transaction ID: 283727

Prediction result:
Anomaly Score: 0.06766200450003801
Prediction: FRAUD
